## Multiprocessing: Producer and Consumer Demonstrations

So far, most of our Python programs have followed one path through the code. One operation happens, then the next, then the next.

**Multiprocessing** allows us to create separate Python processes that can run at the same time.

In this demonstration, we will use a simple producer/consumer example:

- **Marge is the producer.** She makes cookies.
- **Homer is the consumer.** He eats the cookies.

The program provides four different demonstrations:

**Queue**

Marge and Homer communicate through a multiprocessing Queue. Marge puts cookies into the queue and Homer removes them.

**Lock \+ Event**

This version introduces synchronization. A Lock demonstrates how processes can control access to a shared section of code, while an Event signals that Marge has finished producing cookies.

**Pipe \+ ACK**

Instead of a queue, Marge and Homer communicate directly through a Pipe. Homer sends an acknowledgement back after receiving each cookie.

**5 Consumers**

Finally, we keep one producer but create five consumers. All five Homers take cookies from the same queue.

### Running the Demonstrations

Select one of the tabs and click **Start Simulation**.

The **light red window** shows activity from the producer, while the **light green window** shows activity from the consumer or consumers.

Don't worry about understanding every line of the multiprocessing code yet. First, watch what happens in each demonstration. Pay attention to the order of the messages and how that order changes when you run a simulation again.

The important question to keep in mind is:

**How do separate processes communicate and coordinate their work?**



## Building the Multiprocessing Demonstrations

There is quite a bit of code here, but it is really four variations of the same idea. In each demonstration, **Marge produces cookies and Homer consumes them**. What changes is how the processes communicate and coordinate their work.

### The Worker Functions

The functions at the beginning of the program are the functions that will run in separate processes.

The **Queue** example uses:

- queue\_producer() to put cookies into a multiprocessing Queue.
- queue\_consumer() to remove cookies from the Queue.
- An Event lets the consumer know when the producer has finished.

The **Lock \+ Event** example adds a Lock. The with lock: blocks demonstrate how a process can acquire a lock before entering a section of code and automatically release it afterward. The Event still tells Homer when Marge has finished producing cookies.

The **Pipe** example works differently. Instead of putting cookies into a Queue, Marge sends each cookie directly through a Pipe. Homer receives it and sends an acknowledgement back. Marge waits for that response before sending the next cookie.

Finally, the **multiple\-consumer** example has one Marge but five Homers. They all take cookies from the same Queue. This gives us a chance to watch several processes competing for the same work.

### Why Are the Worker Functions Outside the Class?

Notice that the multiprocessing worker functions are defined at the module level rather than as methods of the Tkinter class.

This is especially important when multiprocessing starts new Python processes on Windows. Keeping the worker functions separate also gives us a nice division between the **multiprocessing code** and the **Tkinter interface**.

### Connecting Multiprocessing to Tkinter

The SimulationPanel class builds the interface that we see on each tab.

Each panel has:

- a **Start Simulation** button
- a **Clear Output** button
- a status indicator
- a light red producer window
- a light green consumer window

When **Start Simulation** is clicked, start\_simulation() creates the processes and starts them.

There is also another Queue named output\_queue. This one has a different job. The worker processes put messages into it so that the Tkinter program can display what they are doing.

For example:

("producer", "Marge baked Cookie 1")

tells the GUI to place the message in the producer window, while:

("consumer", "Homer ate Cookie 1")

sends the message to the consumer window.

### Updating the GUI

Tkinter remains responsible for updating its own interface. The worker processes don't directly write into the text windows.

Instead, poll\_output() periodically checks output\_queue for new messages. When it finds one, append\_message() decides whether the message belongs in the producer or consumer window.

The important separation is:

**Worker processes do the work → output Queue carries messages → Tkinter displays the messages**

The call to self.after(75, self.poll\_output) allows Tkinter to check again shortly afterward without sitting in a loop that would prevent the interface from responding.

### When a Simulation Finishes

The program continues checking whether any of its processes are still alive. Once they have all finished, the processes are joined, any remaining output is displayed, and the **Start Simulation** button becomes available again.

terminate\_processes() provides cleanup if the application is closed while processes are still running.

Don't worry about memorizing all of this code. Concentrate on the larger pattern. We have **independent processes doing work**, multiprocessing objects allowing them to **communicate or synchronize**, and Tkinter giving us a way to **watch that activity as it happens**.



In [ ]:
import multiprocessing as mp
import queue
import random
import time
import tkinter as tk
from tkinter import ttk, scrolledtext


# -----------------------------------------------------------------------------
# Worker functions must remain at module level for Windows multiprocessing.
# -----------------------------------------------------------------------------
def queue_producer(cookie_queue, done_event, output_queue, total=10):
    for number in range(1, total + 1):
        time.sleep(random.uniform(0.2, 0.5))
        cookie = f"Cookie {number}"
        cookie_queue.put(cookie)
        output_queue.put(("producer", f"Marge baked {cookie}"))
    done_event.set()
    output_queue.put(("producer", "Marge is done baking!"))


def queue_consumer(cookie_queue, done_event, output_queue):
    while not (done_event.is_set() and cookie_queue.empty()):
        try:
            cookie = cookie_queue.get(timeout=0.5)
            output_queue.put(("consumer", f"Homer ate {cookie}"))
            time.sleep(random.uniform(0.2, 0.6))
        except queue.Empty:
            pass
    output_queue.put(("consumer", "Homer is done eating!"))


def lock_producer(cookie_queue, lock, done_event, output_queue, total=10):
    for number in range(1, total + 1):
        output_queue.put(("producer", "Marge locking"))
        with lock:
            cookie = f"Cookie {number}"
            cookie_queue.put(cookie)
            output_queue.put(("producer", f"Marge baked {cookie}"))
            output_queue.put(("producer", "Marge releasing"))
        time.sleep(random.uniform(0.2, 0.5))
    done_event.set()
    output_queue.put(("producer", "Marge is done baking!"))


def lock_consumer(cookie_queue, lock, done_event, output_queue):
    while not (done_event.is_set() and cookie_queue.empty()):
        cookie = None
        output_queue.put(("consumer", "Homer locking"))
        with lock:
            try:
                cookie = cookie_queue.get_nowait()
            except queue.Empty:
                pass
            if cookie is not None:
                output_queue.put(("consumer", f"Homer ate {cookie}"))
            output_queue.put(("consumer", "Homer releasing"))
        if cookie is not None:
            time.sleep(random.uniform(0.2, 0.6))
        else:
            time.sleep(0.05)
    output_queue.put(("consumer", "Homer is done eating!"))


def pipe_producer(connection, output_queue, total=10):
    for number in range(1, total + 1):
        cookie = f"Cookie {number}"
        output_queue.put(("producer", f"Marge sending cookie: {cookie}"))
        connection.send(cookie)
        output_queue.put(("producer", f"Marge baked {cookie}"))
        output_queue.put(("producer", "Marge waiting for notification"))
        connection.recv()
        time.sleep(0.1)
    connection.send("DONE")
    connection.close()
    output_queue.put(("producer", "Marge is done baking!"))


def pipe_consumer(connection, output_queue):
    while True:
        output_queue.put(("consumer", "Homer waiting"))
        cookie = connection.recv()
        if cookie == "DONE":
            break
        output_queue.put(("consumer", f"Homer ate {cookie}"))
        output_queue.put(("consumer", "Homer sending notification"))
        connection.send("ACK")
        time.sleep(0.1)
    connection.close()
    output_queue.put(("consumer", "Homer is done eating!"))


def pool_producer(cookie_queue, done_event, output_queue, total=50):
    for number in range(1, total + 1):
        cookie = f"Cookie {number}"
        cookie_queue.put(cookie)
        output_queue.put(("producer", f"Marge baked {cookie}"))
        time.sleep(0.2)
    done_event.set()
    output_queue.put(("producer", "Marge is done baking!"))


def pool_consumer(consumer_id, cookie_queue, done_event, output_queue):
    while not (done_event.is_set() and cookie_queue.empty()):
        try:
            cookie = cookie_queue.get(timeout=0.1)
            output_queue.put(("consumer", f"Homer {consumer_id} ate {cookie}"))
            time.sleep(0.2)
        except queue.Empty:
            pass
    output_queue.put(("consumer", f"Homer {consumer_id} is done eating!"))


# -----------------------------------------------------------------------------
# Reusable Tkinter panel
# -----------------------------------------------------------------------------
class SimulationPanel(ttk.Frame):
    def __init__(self, parent, title, description, process_factory,
                 producer_title="Producer (Marge)",
                 consumer_title="Consumer (Homer)"):
        super().__init__(parent, padding=10)
        self.process_factory = process_factory
        self.processes = []
        self.output_queue = None
        self.running = False

        heading = ttk.Label(self, text=title, font=("TkDefaultFont", 15, "bold"))
        heading.pack(anchor="w")
        ttk.Label(self, text=description, wraplength=850).pack(anchor="w", pady=(2, 8))

        controls = ttk.Frame(self)
        controls.pack(fill="x", pady=(0, 8))
        self.start_button = ttk.Button(
            controls, text="Start Simulation", command=self.start_simulation
        )
        self.start_button.pack(side="left")
        ttk.Button(controls, text="Clear Output", command=self.clear_output).pack(
            side="left", padx=8
        )
        self.status = ttk.Label(controls, text="Ready")
        self.status.pack(side="right")

        logs = ttk.Frame(self)
        logs.pack(fill="both", expand=True)
        logs.columnconfigure(0, weight=1)
        logs.columnconfigure(1, weight=1)
        logs.rowconfigure(1, weight=1)

        ttk.Label(logs, text=producer_title).grid(row=0, column=0, pady=(0, 4))
        ttk.Label(logs, text=consumer_title).grid(row=0, column=1, pady=(0, 4))

        self.producer_log = scrolledtext.ScrolledText(
            logs,
            width=42,
            height=22,
            bg="#FFE6E6"          # light red
        )
        self.producer_log.grid(row=1, column=0, sticky="nsew", padx=(0, 5))

        self.consumer_log = scrolledtext.ScrolledText(
            logs,
            width=42,
            height=22,
            bg="#E6F4E6"          # light green
        )
        self.consumer_log.grid(row=1, column=1, sticky="nsew", padx=(5, 0))


    def clear_output(self):
        self.producer_log.delete("1.0", tk.END)
        self.consumer_log.delete("1.0", tk.END)

    def start_simulation(self):
        if self.running:
            return
        self.clear_output()
        self.output_queue = mp.Queue()
        self.processes = self.process_factory(self.output_queue)
        self.running = True
        self.start_button.config(state=tk.DISABLED)
        self.status.config(text="Running")
        for process in self.processes:
            process.start()
        self.after(75, self.poll_output)

    def append_message(self, destination, message):
        target = self.producer_log if destination == "producer" else self.consumer_log
        target.insert(tk.END, message + "\n")
        target.see(tk.END)

    def poll_output(self):
        if self.output_queue is None:
            return
        while True:
            try:
                destination, message = self.output_queue.get_nowait()
                self.append_message(destination, message)
            except queue.Empty:
                break

        if any(process.is_alive() for process in self.processes):
            self.after(75, self.poll_output)
            return

        for process in self.processes:
            process.join(timeout=0.1)

        # One final drain avoids losing messages placed just before a process exits.
        while True:
            try:
                destination, message = self.output_queue.get_nowait()
                self.append_message(destination, message)
            except queue.Empty:
                break

        self.append_message("producer", "Simulation complete.")
        self.append_message("consumer", "Simulation complete.")
        self.running = False
        self.start_button.config(state=tk.NORMAL)
        self.status.config(text="Complete")

    def terminate_processes(self):
        for process in self.processes:
            if process.is_alive():
                process.terminate()
        for process in self.processes:
            process.join(timeout=0.2)

## Creating the Four Multiprocessing Demonstrations

Now that we have the worker functions, we need to create the processes that will run them. Each of these functions sets up one of our four demonstrations.

### Queue

make\_queue\_demo() creates a multiprocessing Queue that can hold up to 10 cookies and an Event to signal when Marge has finished baking.

It creates two processes:

- one running queue\_producer()
- one running queue\_consumer()

Both processes receive the same Queue and Event, allowing them to communicate and determine when the work is finished.

### Lock \+ Event

make\_lock\_demo() again creates a shared Queue and Event, but this time it also creates a Lock.

The producer and consumer receive the same Lock. This lets us demonstrate how a Lock can control access to a section of code so that only one process holds the Lock at a time.

The Event has a different purpose. It signals that Marge has finished producing cookies.

### Pipe \+ ACK

make\_pipe\_demo() replaces the Queue with a multiprocessing Pipe.

A Pipe gives us two connected endpoints. One is passed to the producer and the other to the consumer.

Marge sends a cookie through her end of the Pipe. Homer receives it and sends an acknowledgement back. This creates a back\-and\-forth conversation between the two processes.

### Multiple Consumers

make\_pool\_demo() returns several processes instead of just two.

There is:

- one producer
- five consumers
- one shared Queue
- one shared Event

Marge produces 50 cookies, while the five Homers take cookies from the same Queue. This demonstrates how we can divide work among several consumer processes.

### What to Notice

All four functions return a collection of Process objects, but they connect those processes in different ways.

**Queue:** producer → Queue → consumer

**Lock \+ Event:** producer and consumer share a Queue, Lock, and completion Event

**Pipe:** producer ↔ consumer

**Multiple Consumers:** one producer → Queue → five consumers

These factory functions don't start the processes themselves. They build and return them so that our Tkinter interface can start and monitor each demonstration.



In [ ]:
# -----------------------------------------------------------------------------
# Process factories for the four demonstrations
# -----------------------------------------------------------------------------
def make_queue_demo(output_queue):
    cookie_queue = mp.Queue(maxsize=10)
    done_event = mp.Event()
    return [
        mp.Process(target=queue_producer,
                   args=(cookie_queue, done_event, output_queue, 10)),
        mp.Process(target=queue_consumer,
                   args=(cookie_queue, done_event, output_queue)),
    ]


def make_lock_demo(output_queue):
    cookie_queue = mp.Queue()
    lock = mp.Lock()
    done_event = mp.Event()
    return [
        mp.Process(target=lock_producer,
                   args=(cookie_queue, lock, done_event, output_queue, 10)),
        mp.Process(target=lock_consumer,
                   args=(cookie_queue, lock, done_event, output_queue)),
    ]


def make_pipe_demo(output_queue):
    producer_connection, consumer_connection = mp.Pipe()
    return [
        mp.Process(target=pipe_producer,
                   args=(producer_connection, output_queue, 10)),
        mp.Process(target=pipe_consumer,
                   args=(consumer_connection, output_queue)),
    ]


def make_pool_demo(output_queue):
    cookie_queue = mp.Queue()
    done_event = mp.Event()
    processes = [
        mp.Process(target=pool_producer,
                   args=(cookie_queue, done_event, output_queue, 50))
    ]
    processes.extend(
        mp.Process(target=pool_consumer,
                   args=(number, cookie_queue, done_event, output_queue))
        for number in range(1, 6)
    )
    return processes


## The Multiprocessing Driver

The final part of the program brings everything together.

MultiprocessingDriver creates the main Tkinter window and a ttk.Notebook that gives us four tabs. Each tab contains a SimulationPanel connected to one of the multiprocessing demonstrations we created earlier:

- **Queue** runs make\_queue\_demo()
- **Lock \+ Event** runs make\_lock\_demo()
- **Pipe \+ ACK** runs make\_pipe\_demo()
- **5 Consumers** runs make\_pool\_demo()

The driver doesn't contain the multiprocessing logic itself. Its job is to organize the demonstrations and connect each tab to the appropriate process factory.

Because all four demonstrations use the same SimulationPanel class, we don't have to write a separate GUI for each one. We simply give each panel a title, description, and the function that creates its processes.

### Starting and Closing the Program

The final if \_\_name\_\_ == "\_\_main\_\_": block starts the application.

mp.freeze\_support() provides the multiprocessing startup support needed when the program is run on Windows. We then create the MultiprocessingDriver and start Tkinter's event loop with mainloop().

The driver also handles closing the application. Before destroying the Tkinter window, close\_application() tells each panel to terminate any multiprocessing processes that may still be running.

So the overall structure of our program is:

**Worker functions → Process factories → Simulation panels → Driver**

The worker functions do the multiprocessing work, the factory functions create the processes, the panels display each demonstration, and the driver combines everything into one application.



In [ ]:
# -----------------------------------------------------------------------------
# Composite driver window
# -----------------------------------------------------------------------------
class MultiprocessingDriver(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Multiprocessing Cookie Demonstrations")
        self.geometry("960x620")
        self.minsize(760, 480)

        notebook = ttk.Notebook(self)
        notebook.pack(fill="both", expand=True)

        self.panels = [
            SimulationPanel(
                notebook,
                "Bounded Queue",
                "One producer and one consumer communicate through a bounded multiprocessing Queue.",
                make_queue_demo,
            ),
            SimulationPanel(
                notebook,
                "Lock and Event",
                "A Lock controls access to the shared Queue, while an Event signals completion.",
                make_lock_demo,
            ),
            SimulationPanel(
                notebook,
                "Pipe Handshake",
                "A duplex Pipe sends each cookie and returns an acknowledgement before continuing.",
                make_pipe_demo,
            ),
            SimulationPanel(
                notebook,
                "Multiple Consumers",
                "One producer places 50 cookies in a Queue shared by five consumer processes.",
                make_pool_demo,
                consumer_title="Consumers (Homers 1-5)",
            ),
        ]

        labels = ("Queue", "Lock + Event", "Pipe + ACK", "5 Consumers")
        for panel, label in zip(self.panels, labels):
            notebook.add(panel, text=label)

        self.protocol("WM_DELETE_WINDOW", self.close_application)

    def close_application(self):
        for panel in self.panels:
            panel.terminate_processes()
        self.destroy()


if __name__ == "__main__":
    mp.freeze_support()
    app = MultiprocessingDriver()
    app.mainloop()

## Multiprocessing Demo Summary

In this demonstration, we used the same basic producer/consumer problem to look at several ways Python processes can work together.

**Queue** showed us how one process can produce data while another process consumes it. The Queue provides a place for information to be passed safely between the processes.

**Lock \+ Event** introduced synchronization. The Lock demonstrated how access to a section of code can be controlled, while the Event gave the producer a way to signal that it had finished its work.

**Pipe \+ ACK** demonstrated direct communication between two processes. Marge sent a cookie to Homer, Homer received it, and then sent an acknowledgement back before Marge continued.

**Multiple Consumers** expanded the idea beyond two processes. One producer created the work while five consumers shared that work by taking cookies from the same Queue.

The Tkinter interface gave us a way to watch all of this happen. The producer activity appeared in **light red** and the consumer activity appeared in **light green**, making it easier to see how the processes operated independently.

### The Big Picture

There isn't just one way for processes to work together.

- A **Queue** is useful for passing data between processes.
- A **Lock** can control access when processes need synchronization.
- An **Event** can signal that something has happened.
- A **Pipe** provides direct communication between processes.
- Multiple processes can share a workload rather than having one process do everything.

The important idea is not to memorize all of the multiprocessing syntax. Instead, understand why we might choose one of these techniques and watch how changing the communication method changes the behavior of the program.

**Multiprocessing lets us divide work among separate processes, but once we do that, we also need a way for those processes to communicate and coordinate with one another.**

